# Azure Demo Candlestick Run

This notebook is designed for Azure Jupyter environments. It installs dependencies, clears stale run artifacts, runs the balanced demo experiment flow, builds the dashboard snapshot, and prints the resulting metrics artifacts.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "configs" / "config.yaml").exists():
    raise RuntimeError("Open this notebook from the repository root in Azure.")

print(f"Repo root: {REPO_ROOT}")
print(f"Python: {sys.executable}")

In [ ]:
%pip install -r requirements.txt

In [ ]:
import subprocess
import shutil

RUN_NAME = "azure_demo_balanced_run"
BASE_CONFIG = "configs/config.yaml"
TRAINING_OVERRIDES = [
    "configs/overrides/production_repro.yaml",
    "configs/overrides/demo_balanced.yaml",
]
CLEAN_RUN_ARTIFACTS = True

def clean_run_artifacts(run_name: str):
    targets = [
        REPO_ROOT / "outputs" / "metrics" / run_name,
        REPO_ROOT / "outputs" / "checkpoints" / run_name,
        REPO_ROOT / "outputs" / "backtest" / run_name,
        REPO_ROOT / "outputs" / "gallery" / run_name,
        REPO_ROOT / "web-dashboard" / ".generated" / run_name,
    ]
    for target in targets:
        if target.exists():
            shutil.rmtree(target)
            print(f"Removed stale artifact: {target}")

def run_script(script: str, extra_args: list[str] | None = None, config_overrides: list[str] | None = None):
    cmd = [sys.executable, script, "--config", BASE_CONFIG]
    for override in config_overrides or []:
        cmd.extend(["--config-override", override])
    cmd.extend(["--set", f"project.run_name={RUN_NAME}"])
    cmd.extend(extra_args or [])
    print("$", " ".join(cmd))
    subprocess.run(cmd, cwd=REPO_ROOT, check=True)


In [ ]:
processed_prices = REPO_ROOT / "data" / "processed" / "spy_15m.csv"

if processed_prices.exists():
    print(f"Using existing processed dataset: {processed_prices}")
else:
    print("No processed dataset found. Running data preparation first.")
    run_script("scripts/download_kaggle_intraday.py")
    run_script("scripts/prepare_15m_dataset.py")

if CLEAN_RUN_ARTIFACTS:
    clean_run_artifacts(RUN_NAME)

run_script("scripts/run_experiment_suite.py", config_overrides=TRAINING_OVERRIDES)
run_script("scripts/run_backtest.py", config_overrides=TRAINING_OVERRIDES)
run_script("scripts/render_pattern_gallery.py", config_overrides=TRAINING_OVERRIDES)
run_script(
    "scripts/build_results_dashboard.py",
    extra_args=["--run-name", RUN_NAME, "--output-root", "web-dashboard/.generated", "--data-only"],
)


In [ ]:
import json
import pandas as pd
from IPython.display import display

metrics_root = REPO_ROOT / "outputs" / "metrics" / RUN_NAME
dashboard_snapshot = REPO_ROOT / "web-dashboard" / ".generated" / RUN_NAME / "data.json"

summary = pd.read_csv(metrics_root / "model_comparison_summary.csv")
champions = pd.read_csv(metrics_root / "champions.csv")
macro = json.loads((metrics_root / "macro_summary.json").read_text(encoding="utf-8"))
label_sanity = {
    pattern_dir.name: json.loads((pattern_dir / "label_sanity.json").read_text(encoding="utf-8"))
    for pattern_dir in metrics_root.iterdir()
    if pattern_dir.is_dir() and (pattern_dir / "label_sanity.json").exists()
}

print("Macro summary:")
print(json.dumps(macro, indent=2))
print(f"Dashboard snapshot: {dashboard_snapshot}")
print("\nLabel sanity summaries:")
for pattern, payload in sorted(label_sanity.items()):
    summary_payload = payload.get("dataset_summary", {})
    print(
        f"- {pattern}: samples={summary_payload.get('samples', 0)}, "
        f"positives={summary_payload.get('positive_labels', 0)}, "
        f"negatives={summary_payload.get('negative_labels', 0)}"
    )

display(summary)
display(champions)
